# S3 — Temporal reweighting (multi-seed)

`YEAR_WEIGHTS = {2016: 1.0, 2015: 0.75, 2014: 0.5}`. Seeds 42, 1, 2.

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))


In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score
from huggingface_hub import snapshot_download
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed,
)
from datasets import Dataset

DATA_DIR = Path(snapshot_download(repo_id='tamarasuarezrod/longeval-data', repo_type='dataset'))

MODEL_NAME   = 'roberta-base'
SEEDS        = [42, 1, 2]
STRATEGY     = 's3-temporal-reweighting'
YEAR_WEIGHTS = {2016: 1.0, 2015: 0.75, 2014: 0.5}
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Device: cuda


## Data

In [3]:
def load_split(path, label_col='label'):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={label_col: 'label'})
    return df[['pp_text', 'label']]

def load_train(path):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={'distant_label': 'label'})
    df['year'] = df['created_at'].str.strip().str[-4:].astype(int)
    df['sample_weight'] = df['year'].map(YEAR_WEIGHTS).fillna(1.0)
    return df[['pp_text', 'label', 'sample_weight']]

train_df = load_train(DATA_DIR / 'train_eval/train.json')
eval_df  = load_split(DATA_DIR / 'train_eval/interim_eval_2016.json', label_col='distant_label')
test_dfs = {
    'test_within': load_split(DATA_DIR / 'test/interim_test_2016.json'),
    'test_short':  load_split(DATA_DIR / 'test/interim_test_2018.json'),
    'test_long':   load_split(DATA_DIR / 'test/interim_test_2021.json'),
}

label2id = {l: i for i, l in enumerate(sorted(train_df['label'].unique()))}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print('Labels:', label2id)


Labels: {'negative': 0, 'positive': 1}


## Training loop (seeds 42, 1, 2)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_train_dataset(df):
    d = Dataset.from_dict({'text': df['pp_text'].tolist(),
                           'label': df['label'].map(label2id).tolist(),
                           'sample_weight': df['sample_weight'].tolist()})
    return d.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
                 batched=True, remove_columns=['text'])

def make_dataset(df):
    d = Dataset.from_dict({'text': df['pp_text'].tolist(),
                           'label': df['label'].map(label2id).tolist()})
    return d.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
                 batched=True, remove_columns=['text'])

train_ds = make_train_dataset(train_df)
eval_ds  = make_dataset(eval_df)
test_ds  = {n: make_dataset(df) for n, df in test_dfs.items()}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        w = inputs.pop('sample_weight', None)
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        loss = loss_fct(outputs.logits, labels)
        if w is not None:
            loss = (loss * w.to(loss.device)).mean()
        else:
            loss = loss.mean()
        return (loss, outputs) if return_outputs else loss

all_results  = {}
saved_models = {}

for SEED in SEEDS:
    print(f'\n{"="*50}  SEED {SEED}')
    set_seed(SEED)
    models_dir = Path(f'/tmp/s3_seed{SEED}')
    models_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=str(models_dir), num_train_epochs=25,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=2e-05, warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
        logging_steps=50, fp16=(DEVICE=='cuda'), seed=SEED, report_to='none',
        remove_unused_columns=False)

    trainer = WeightedTrainer(model=model, args=args,
                              train_dataset=train_ds, eval_dataset=eval_ds,
                              processing_class=tokenizer,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()

    seed_results = {}
    for name, ds in test_ds.items():
        preds_out = trainer.predict(ds)
        f1 = f1_score(np.array(ds['label']), np.argmax(preds_out.predictions, axis=1), average='macro')
        seed_results[name] = f1
        print(f'  {name}: F1={f1:.4f}')

    all_results[SEED]  = seed_results
    saved_models[SEED] = trainer.model


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

  test_within: F1=0.7321


  test_short: F1=0.6661


  test_long: F1=0.7058


## Summary

In [5]:
# ── Summary ────────────────────────────────────────────────
import numpy as np

splits = ['test_within', 'test_short', 'test_long']
print(f"{'seed':>8}  {'within':>7}  {'short':>7}  {'long':>7}  {'RPD_short':>10}  {'RPD_long':>9}")
for seed, r in all_results.items():
    rpd_s = (r['test_short']  - r['test_within']) / r['test_within']
    rpd_l = (r['test_long']   - r['test_within']) / r['test_within']
    print(f"{seed:>8}  {r['test_within']:>7.4f}  {r['test_short']:>7.4f}  {r['test_long']:>7.4f}  {rpd_s:>+10.4f}  {rpd_l:>+9.4f}")

print()
for sp in splits:
    vals = [all_results[s][sp] for s in all_results]
    print(f"{sp}: mean={np.mean(vals):.4f}  std={np.std(vals,ddof=1):.4f}  [{min(vals):.4f}–{max(vals):.4f}]")


    seed   within    short     long   RPD_short   RPD_long
      42   0.7299   0.6849   0.6936     -0.0616    -0.0497
       1   0.7362   0.6883   0.7047     -0.0651    -0.0428
       2   0.7321   0.6661   0.7058     -0.0902    -0.0360

test_within: mean=0.7327  std=0.0032  [0.7299–0.7362]
test_short: mean=0.6798  std=0.0120  [0.6661–0.6883]
test_long: mean=0.7013  std=0.0067  [0.6936–0.7058]


## Save to HuggingFace

In [6]:
for seed, model in saved_models.items():
    repo = f'tamarasuarezrod/longeval-{STRATEGY}-seed{seed}'
    model.push_to_hub(repo)
    tokenizer.push_to_hub(repo)
    print('Saved:', repo)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-s3-temporal-reweighting-seed42


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-s3-temporal-reweighting-seed2
